In [0]:
import tmdbsimple as tmdb
import os
import requests
import zipfile
import time
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
tmdb.API_KEY = dbutils.secrets.get(scope='movie-recommender',key='API_KEY')
tmdb.REQUESTS_TIMEOUT = (2, 5)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS movie_recommender

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS movie_recommender.bronze

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS movie_recommender.bronze.movielens

In [0]:
FILE_PATH='/Volumes/movie_recommender/bronze/movielens'

In [0]:
def file_exists_in_path(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception as e:
        if 'java.io.FileNotFoundException' in str(e):
            return False
        else:
            return e

In [0]:
def get_data_from_movielens(url,FILE_PATH,filename):
    os.makedirs(f'{FILE_PATH}/zip',exist_ok=True)
    os.makedirs(f'{FILE_PATH}/csv',exist_ok=True)
    zip_file_path=f'{FILE_PATH}/zip/{filename}.zip'
    files_needed=['tags','links','movies','ratings']
    req=requests.get(url,stream=True)
    if req.status_code==200:
        with open(zip_file_path,'wb') as f:
            for chunk in req.iter_content(chunk_size=8192):
                f.write(chunk)
    with zipfile.ZipFile(zip_file_path,'r') as z:
        z.extractall(f'{FILE_PATH}/zip/')
    for file in files_needed:
        if file_exists_in_path(f'{FILE_PATH}/zip/{filename}/{file}.csv'):
            table=spark.read.option('header',True).option("quote", '"').option("escape", '"').option("multiLine", "true").csv(f'{FILE_PATH}/zip/{filename}/{file}.csv')
            table.write.format("delta").mode("overwrite").saveAsTable(f'movie_recommender.bronze.{file}')
            print(f'Wrote {file} successfully')
        else:
            print(f'{file}.csv does not exist in path.')
    dbutils.fs.mv(f'{FILE_PATH}/zip/{filename}',f'{FILE_PATH}/csv',recurse=True)
    dbutils.fs.rm(f'{FILE_PATH}/zip',True)
get_data_from_movielens('https://files.grouplens.org/datasets/movielens/ml-32m.zip',FILE_PATH,'ml-32m')